Our package provides data access in a Python programming environment.

Here, we will start a Clustering analysis for the Pancreatic ductal adenocarcinoma (pdac).

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

# from gpnotebook.tools.standard_imports import *
import os, re,sys
import yaml
import pandas as pd
import numpy as np


In [2]:
# project_dir = r"/Users/yingweihu/Documents/GitHub/glycoproteinnotebook-private/data/v1/projects/PDAC_P_PDC000271"
project_dir = r"/Users/yingweihu/Documents/GitHub/glycoproteinnotebook-private/data/v1/projects/GBM_P_PDC000205"
data_dir = os.path.join(project_dir,"matrix")
meta_dir = os.path.join(project_dir,"meta")
job_dir = os.path.join(project_dir,"precomputed","cluster")
if not os.path.exists(job_dir):
    os.mkdir(job_dir)

In [3]:
data_path = os.path.join(data_dir, "DIG_nglycoform-peptide_matrix-abundances-MD_norm.tsv")
data_df = pd.read_csv(data_path,sep="\t", index_col = [0,1,2,3])
data_df

Intensity.Reference  \
Site                                               Gene     Sequence                 Glycan                             
ENSP00000300107@452                                CLPX     AAAAADLANR               N5H4F1S0G0             11.025784   
ENSP00000349437@2122                               IGF2R    AACAVKPQEVQMVNGTITNPINGK N4H5F1S2G0             12.964216   
                                                                                     N5H6F3S1G0             11.104464   
ENSP00000418081@773;ENSP00000407393@773;ENSP000... CACNA2D2 AAEDWTENPEPFNASFYR       N2H10F0S0G0            13.972249   
ENSP00000353032@196;ENSP00000336607@180            P2RX4    AAENFTLLVK               N3H6F0S1G0             13.389903   
...                                                                                                               ...   
ENSP00000376793@267;ENSP00000451119@49             SERPINA3 YTGNASALFILPDQDK         N7H8F0S3G0             12.148197   
                                                                                     N7H8F2S4G0             13.734268   
                                                                                     N7H8F3S4G0             10.416718   
                                                                                     N7H8F5S2G0             11.224668   
                                                                                     N8H10F1S1G0            11.053432   

                                                                                                    pool_01  \
Site                                               Gene     Sequence                 Glycan                   
ENSP00000300107@452                                CLPX     AAAAADLANR               N5H4F1S0G0   11.025784   
ENSP00000349437@2122                               IGF2R    AACAVKPQEVQMVNGTITNPINGK N4H5F1S2G0   12.964216   
                                                                                     N5H6F3S1G0   11.104464   
ENSP00000418081@773;ENSP00000407393@773;ENSP000... CACNA2D2 AAEDWTENPEPFNASFYR       N2H10F0S0G0  13.972249   
ENSP00000353032@196;ENSP00000336607@180            P2RX4    AAENFTLLVK               N3H6F0S1G0   13.389903   
...                                                                                                     ...   
ENSP00000376793@267;ENSP00000451119@49             SERPINA3 YTGNASALFILPDQDK         N7H8F0S3G0         NaN   
                                                                                     N7H8F2S4G0         NaN   
                                                                                     N7H8F3S4G0         NaN   
                                                                                     N7H8F5S2G0         NaN   
                                                                                     N8H10F1S1G0        NaN   

                                                                                                  GTEX-Y8DK-0011-R10A-SM-HAKY1_N_01  \
Site                                               Gene     Sequence                 Glycan                                           
ENSP00000300107@452                                CLPX     AAAAADLANR               N5H4F1S0G0                           12.956685   
ENSP00000349437@2122                               IGF2R    AACAVKPQEVQMVNGTITNPINGK N4H5F1S2G0                           11.551754   
                                                                                     N5H6F3S1G0                           11.545444   
ENSP00000418081@773;ENSP00000407393@773;ENSP000... CACNA2D2 AAEDWTENPEPFNASFYR       N2H10F0S0G0                          16.819940   
ENSP00000353032@196;ENSP00000336607@180            P2RX4    AAENFTLLVK               N3H6F0S1G0                           13.509790   
...                                                                                                                             ...   


In [4]:
meta_path= os.path.join(meta_dir, "GBM_meta.txt")
meta_df = pd.read_csv(meta_path,sep="\t",header=[0,1])
meta_df

,case_id,Age,Sex,Tumor_Size_cm,Histologic_Grade,Tumor_necrosis,Path_Stage_pT,Path_Stage_pN,Stage,BMI,Tobacco_smoking_history,ATRX_mutation,PIK3CA_mutation,RB1_mutation,TP53_mutation,EGFR_mutation,PTEN_mutation
,data_type,CON,BIN,CON,ORD,BIN,ORD,ORD,ORD,CON,ORD,BIN,BIN,BIN,BIN,BIN,BIN
0,C3L-00104,58,Male,1.0,NaN,NaN,NaN,NaN,NaN,32.54,past smoker,0,0,1,1,0,1
1,C3L-00365,59,Female,1.2,NaN,NaN,NaN,NaN,NaN,20.61,past smoker,0,0,1,1,1,0
2,C3L-00674,45,Male,4.9,NaN,NaN,NaN,NaN,NaN,27.44,NaN,0,0,0,0,1,1
3,C3L-00677,69,Female,4.5,NaN,NaN,NaN,NaN,NaN,19.32,current smoker,1,0,1,1,0,1
4,C3L-01040,77,Male,5.0,NaN,NaN,NaN,NaN,NaN,24.22,NaN,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,C3N-03183,53,Male,7.0,NaN,NaN,NaN,NaN,NaN,26.12,non-smoker,0,0,0,0,1,1
95,C3N-03184,35,Male,3.5,NaN,NaN,NaN,NaN,NaN,26.30,current smoker,0,0,0,0,0,0
96,C3N-03186,54,Female,8.0,NaN,NaN,NaN,NaN,NaN,21.63,non-smoker,0,1,0,0,0,0


In [5]:
meta_cols = ['case_id','Sex','Stage']
meta2 = meta_df.loc[:,meta_cols]
meta2.columns = ['Sample.ID'] + meta_cols[1:]
meta2

,Sample.ID,Sex,Stage
0,C3L-00104,Male,NaN
1,C3L-00365,Female,NaN
2,C3L-00674,Male,NaN
3,C3L-00677,Female,NaN
4,C3L-01040,Male,NaN
...,...,...,...
94,C3N-03183,Male,NaN
95,C3N-03184,Male,NaN
96,C3N-03186,Female,NaN
97,C3N-03188,Male,NaN


In [6]:
meta2.head(17)

,Sample.ID,Sex,Stage
0,C3L-00104,Male,NaN
1,C3L-00365,Female,NaN
2,C3L-00674,Male,NaN
3,C3L-00677,Female,NaN
4,C3L-01040,Male,NaN
5,C3L-01043,Male,NaN
6,C3L-01045,Female,NaN
7,C3L-01046,Male,NaN
8,C3L-01048,Male,NaN
9,C3L-01049,Male,NaN


In [7]:
head_cols = ['Site', 'Gene', 'Sequence', 'Glycan', 'Intensity.Reference']
samples = [i for i in data_df.columns.values if i not in head_cols]
samples = [i for i in samples if i.split('_')[0] in list(meta2['Sample.ID']) and i.split('_')[1] == 'T']
len(samples)

99

In [8]:
rows = []
for sample in samples:
    key = sample.split('_')[0]
    row = meta2[meta2['Sample.ID']==key].iloc[0]
    row['Sample.ID'] = sample
    rows.append(row)
meta3 = pd.DataFrame(rows)

In [9]:
meta3.head(17)

,Sample.ID,Sex,Stage
94,C3N-03183_T_01,Male,NaN
61,C3N-01505_T_01,Male,NaN
97,C3N-03188_T_01,Male,NaN
33,C3L-02984_T_01,Male,NaN
24,C3L-02542_T_01,Female,NaN
11,C3L-01142_T_01,Female,NaN
19,C3L-01834_T_01,Female,NaN
81,C3N-02256_T_01,Male,NaN
66,C3N-01814_T_01,Female,NaN
83,C3N-02770_T_02,Male,NaN


In [10]:
meta3 = meta3.replace(np.nan,'NA')

In [11]:
top_ann_data_path = os.path.join(job_dir,'top_ann_data.tsv')
meta3.to_csv(top_ann_data_path, sep="\t", index=False)

Top annotation settings.

In [12]:

top_ann_settings = {
    'Sex': {
        'Male': 'blue',
        'Female': 'red',
        'NA': 'grey',
    },
    'Stage': {
        'Stage I': 'blue',
        'Stage II': 'green',
        'Stage III': 'orange',
        'Stage IV': 'red',
        'NA': 'grey'
    },

}
top_ann_settings_path = os.path.join(job_dir,'top_ann_settings.yml')
with open(top_ann_settings_path,'w') as f:
    yaml.dump(top_ann_settings,f,default_flow_style=False)

In [13]:
data_df.head(2)

,,,,Intensity.Reference,pool_01,GTEX-Y8DK-0011-R10A-SM-HAKY1_N_01,C3N-03183_T_01,C3N-01505_T_01,C3N-03188_T_01,C3L-02984_T_01,C3L-02542_T_01,C3L-01142_T_01,C3L-01834_T_01,...,C3L-01327_T_11,C3N-02786_T_11,C3L-01040_T_11,C3N-02188_T_11,C3N-01515_T_11,C3L-00104_T_11,C3N-02769_T_11,C3N-02785_T_11,GTEX-NPJ7-0011-R10A-SM-HAKXW_N_11,C3N-02190_T_11
Site,Gene,Sequence,Glycan,,,,,,,,,,,,,,,,,,,,,
ENSP00000300107@452,CLPX,AAAAADLANR,N5H4F1S0G0,11.025784,11.025784,12.956685,11.897541,11.282224,12.863955,11.621552,13.101060,12.629349,9.471347,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ENSP00000349437@2122,IGF2R,AACAVKPQEVQMVNGTITNPINGK,N4H5F1S2G0,12.964216,12.964216,11.551754,13.739682,13.105198,13.060432,12.311538,12.543199,13.943684,12.959484,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
data_df.shape

(28285, 122)

In [15]:
samples = meta3['Sample.ID'].to_list()

In [16]:
len(samples)

99

In [17]:
df2 = data_df.loc[:,samples].dropna()

In [18]:
df2.shape

(298, 99)

In [19]:
from scipy.stats import variation
rows = []
for index,row in df2.iterrows():
    rows.append([variation([np.power(2,i) for i in list(row)])])
cv_df = pd.DataFrame(rows,columns=['cv'],index= df2.index)

glycopeptides = cv_df[cv_df['cv']>0.25].index

data2 = df2[df2.index.isin(glycopeptides)]
glycopeptides =  [f'{site}@{gene}@{seq}@{glycan}' for site,gene,seq,glycan in glycopeptides]
data2.index = glycopeptides
tumor_expression_path = os.path.join(job_dir,'expression_data.tsv')
data2.to_csv(tumor_expression_path,sep='\t',index=True)

In [20]:
data2.shape

(294, 99)

Extract tumor samples from glycopeptide expression data based on pathological status,

calculates the coefficient of variation (CV) for each glycopeptide, selects glycopeptides with CV greater than 0.25.

Map glcopeptides with cv>0.25 in tumor patients with glycan type.

In [21]:
import re,os, sys

def decide_glycan_type(g):
    m = re.finditer("([A-Z])([\d]+)", g)
    y = [(i.group(1), int(i.group(2))) for i in m]
    d = dict(y)
    glycan_type = "Other"
    if d["N"] == 2 and d["H"] >= 5 and d["F"] == 0 and d["S"] == 0 and d["G"] == 0:
        glycan_type = "HM"
    elif d["N"] >= 2 and d["H"] >= 3 and d["F"] > 0 and d["S"] == 0:
        glycan_type = "only_F"
    elif d["N"] >= 2 and d["H"] >= 3 and d["S"] > 0 and d["F"] == 0:
        glycan_type = "only_S"
    elif d["N"] >= 2 and d["H"] >= 3 and d["S"] > 0 and d["F"] > 0:
        glycan_type = "F+S"
    return glycan_type


In [22]:
# left annotation
# from gpnotebook.tools.glycan import decide_glycan_type

glycan_type_map = dict(zip(glycopeptides,[decide_glycan_type(i) for i in glycopeptides]))
  
left_ann_data_path =  os.path.join(job_dir,'left_annotation_data.tsv')
rows = []
for i in glycan_type_map:
    rows.append([i,glycan_type_map[i]])
left_ann_data = pd.DataFrame(rows,columns=['Glycopeptide','GlycanType'])
left_ann_data.to_csv(left_ann_data_path,sep="\t",index=False)

In [23]:
left_ann_data

,Glycopeptide,GlycanType
0,ENSP00000262776@541@LGALS3BP@AAIPSALDTNSSK@N3H...,F+S
1,ENSP00000273784@166;ENSP00000393887@165@AHSG@A...,only_S
2,ENSP00000345179@82@APOD@ADGTVNQIEGEATPVNLTEPAK...,only_S
3,ENSP00000345179@82@APOD@ADGTVNQIEGEATPVNLTEPAK...,F+S
4,ENSP00000345179@82@APOD@ADGTVNQIEGEATPVNLTEPAK...,only_S
...,...,...
289,ENSP00000308541@135;ENSP00000433907@135@F2@YPH...,F+S
290,ENSP00000376793@267;ENSP00000451119@49@SERPINA...,only_S
291,ENSP00000376793@267;ENSP00000451119@49@SERPINA...,F+S
292,ENSP00000376793@267;ENSP00000451119@49@SERPINA...,only_S


Map glycan types with colors.

In [24]:

# left annotation settings, including color, order
left_ann_settings_path = os.path.join(job_dir,'left_annotation_settings.yml')
left_ann_settings = {
    "glycan_type_index" :{
    "HM": 1,
    "only_F":2,
    "only_S":3,
    "F+S":4,
    "Other":5
    },
    "glycan_type_color" : {
        "HM": 'green',
    "only_F": 'red',
    "only_S": 'purple',
    "F+S": 'orange',
    "Other": 'grey'
}
}
with open(left_ann_settings_path,'w') as f:
    yaml.dump(left_ann_settings,f,default_flow_style=False)
    

Parameters for NMF clustering.

In [25]:
nmf_parameters_path = os.path.join(job_dir, 'nmf_parameters.yml')
nmf_parameters = {
    'k_range': {
        'min': 3,
        'max': 5,
    },
    'test':{
        'nruns': 50
    },
    'opt_k':{
        'nruns': 500,
        'predefined': 0,
        'value': 4,
        'feature_prob': 0.8
    }
}
with open(nmf_parameters_path,'w') as f:
    yaml.dump(nmf_parameters,f,default_flow_style=False)

Generate a YAML configuration file (nmf_configs.yml) containing paths to various data required for NMF clustering.

In [26]:
config_data = {
    'input': {
        'expression_data': tumor_expression_path,
        'left_annotation_data': left_ann_data_path ,
        'left_annotation_settings': left_ann_settings_path,
        'top_annotation_data': top_ann_data_path,
        'top_annotatin_settings': top_ann_settings_path,
        'nmf_parameters': nmf_parameters_path
    },
    'output':{
        'out_dir': job_dir
    }
}
nmf_configs_path = os.path.join(job_dir,'nmf_configs.yml')
with open(nmf_configs_path,'w') as f:
    yaml.dump(config_data,f,default_flow_style=False)